# Multivariate Linear Regression from Scratch

This project predicts house prices using the California Housing dataset.
The model is built with Python and NumPy only, and trained with gradient descent.

## 1. Load the Data

We load the California Housing dataset from scikit-learn.
`x` contains the features (8 columns) and `y` contains the house prices.

In [13]:
import numpy as np
from sklearn.datasets import fetch_california_housing
data=fetch_california_housing()
x=data.data
y=data.target

## 2. Split the Data

We shuffle the data randomly (with a fixed seed so the result is repeatable).
Then we split it into:
- 16,512 samples for training
- 4,128 samples for testing

In [14]:
indices=np.arange(20640)
np.random.seed(0)
shuffled_indices=np.random.permutation(indices)
x_shuffled=x[shuffled_indices]
y_shuffled=y[shuffled_indices]
x_train=x_shuffled[:16512]
y_train=y_shuffled[:16512]
x_test=x_shuffled[16512:]
y_test=y_shuffled[16512:]

## 3. Model Functions

- `model_function`: predicts the price using the dot product of features and weights, plus the bias.
- `cost_function`: measures the error using the sum of squared errors divided by 2m.
- `gradient_descent`: calculates the gradients of the weights (`dw`) and the bias (`db`).

In [15]:
w=np.array([0.1,-0.5,0.2,0.4,-0.2,1,2,3])
b=0.5
def model_function(x,w,b):
    prediction=np.dot(x,w)+b
    return prediction
def cost_function(x,y,w,b):
    cost=0
    prediction=model_function(x,w,b)
    error=prediction-y
    error_squared=np.sum(error**2)
    cost=error_squared/(2*len(x))
    return cost
def gradient_descent(x,y,w,b):
    prediction=model_function(x,w,b)
    error=prediction-y
    error=error.reshape(-1,1)
    erx=error*x
    dw=np.sum(erx,axis=0)
    db=np.sum(error)
    dw=dw/len(x)
    db=db/len(x)
    return dw,db

## 4. Training

We train the model with gradient descent.
- Learning rate: 0.0000005 (a larger value caused overflow)
- Maximum iterations: 2,500,000
- Training stops when the cost improvement is smaller than the threshold (0.0000001)

Training stopped after about 2.37 million iterations.
Final training cost: about 0.359.

In [2]:
alpha=0.0000005
previous_cost=cost_function(x_train,y_train,w,b)
num_iters=2500000
threshold=0.0000001
for i in range(num_iters):
    dw,db=gradient_descent(x_train,y_train,w,b)
    w=w-alpha*dw
    b=b-alpha*db
    current_cost=cost_function(x_train,y_train,w,b)
    improvement=previous_cost-current_cost
    previous_cost=current_cost
    if improvement<=threshold:
        break
print(w)
print(b)
print(current_cost)
print(i)
print(improvement)
    
    

[ 5.27510968e-01  1.93054997e-02 -1.21227767e-01  3.97789438e-01
  5.01820547e-05 -4.87151948e-03  1.24852807e-01  4.48159467e-02]
0.5472857482097983
0.35899230305913443
2379480
9.999991318343504e-08


## 5. Evaluation on Test Data

We test the trained model on data it has never seen.

Note: here the cost is divided by `m` (not `2m`), so the result is about 0.667.
With the same formula as training (divided by `2m`), the test cost is about 0.333.
This is close to the training cost (0.359), so the model generalizes well.

In [12]:
y_simple=y_test
x_simple=x_test
w=np.array([ 5.27510968e-01,1.93054997e-02,-1.21227767e-01,3.97789438e-01,5.01820547e-05,-4.87151948e-03,1.24852807e-01,4.48159467e-02])
b=0.5472857482097983
def model_function(x,w,b):
    prediction=np.dot(x_simple,w)+b
    return prediction
def cost_function(x,y,w,b):
    prediction=model_function(x,w,b)
    error=prediction-y
    squared_error=np.sum(error**2)
    cost=squared_error/len(x)
    return(cost)
result=cost_function(x_simple,y_simple,w,b)
print(result)



0.6668472834427891


## 6. Comparison with the Exact Solution

We compare gradient descent with the exact least-squares solution (`np.linalg.lstsq`).

- Gradient descent training cost: about 0.358
- Least-squares training cost: about 0.265

Gradient descent stopped early. The features are not scaled, so a very small
learning rate was needed, and the improvement per step became smaller than
the threshold before reaching the minimum.
Feature scaling would likely fix this.

In [16]:
def cost(x, y, w, b):
    return np.sum((x @ w + b - y)**2) / (2*len(x))

x_b = np.c_[x_train, np.ones(len(x_train))]
theta = np.linalg.lstsq(x_b, y_train, rcond=None)[0]
w_ls, b_ls = theta[:-1], theta[-1]

print(w_ls)
print(b_ls)
print(cost(x_train, y_train, w_ls, b_ls))
print(cost(x_test, y_test, w_ls, b_ls))

[ 4.29919231e-01  9.16865448e-03 -1.00544366e-01  5.95326628e-01
 -7.98499668e-06 -3.38289828e-03 -4.25576132e-01 -4.37969270e-01]
-37.14515723690773
0.2658515770230842
0.24784412028829314
